In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
import json

with open("../data/processed/top_sensors.json") as f:
    top_sensors = json.load(f)

print(top_sensors)

['sensor_11', 'sensor_9', 'sensor_4', 'sensor_12', 'sensor_7', 'sensor_14', 'sensor_15', 'sensor_21', 'sensor_13', 'sensor_2']


In [3]:
df = pd.read_csv("../data/processed/train_scaled.csv")
print(df.shape)
df.head()

(20631, 23)


,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,...,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,-1.721725,-0.134255,-0.925936,-5.329071e-15,0.141683,...,-0.266467,0.334262,-1.058890,-0.269071,-0.603816,-3.469447e-18,-0.781710,1.348493,1.194427,125
1,1,2,0.0019,-0.0003,100.0,-1.061780,0.211528,-0.643726,-5.329071e-15,0.141683,...,-0.191583,1.174899,-0.363646,-0.642845,-0.275852,-3.469447e-18,-0.781710,1.016528,1.236922,125
2,1,3,-0.0043,0.0003,100.0,-0.661813,-0.413166,-0.525953,-5.329071e-15,0.141683,...,-1.015303,1.364721,-0.919841,-0.551629,-0.649144,-3.469447e-18,-2.073094,0.739891,0.503423,125
3,1,4,0.0007,0.0000,100.0,-0.661813,-1.261314,-0.784831,-5.329071e-15,0.141683,...,-1.539489,1.961302,-0.224597,-0.520176,-1.971665,-3.469447e-18,-0.781710,0.352598,0.777792,125
4,1,5,-0.0019,-0.0002,100.0,-0.621816,-1.251528,-0.301518,-5.329071e-15,0.141683,...,-0.977861,1.052871,-0.780793,-0.521748,-0.339845,-3.469447e-18,-0.136018,0.463253,1.059552,125


In [4]:
def create_windows(df, sensors, window_size):
    
    X = []
    y = []

    for engine_id in df["engine_id"].unique():
        
        engine_df = df[df["engine_id"] == engine_id]
        engine_df = engine_df.sort_values("cycle")

        sensor_values = engine_df[sensors].values
        rul_values = engine_df["RUL"].values

        for i in range(len(engine_df) - window_size + 1):
            
            window = sensor_values[i:i+window_size]
            target = rul_values[i+window_size-1]

            X.append(window.flatten())
            y.append(target)

    return np.array(X), np.array(y)

In [5]:
window_sizes = [5,10,20,30,40]

output_dir = Path("../data/processed/windows")
output_dir.mkdir(parents=True, exist_ok=True)

for w in window_sizes:

    print(f"Generating window size {w}")

    X, y = create_windows(df, top_sensors, w)

    save_path = output_dir / f"window_{w}.npz"

    np.savez(save_path, X=X, y=y)

    print("Shape:", X.shape)

Generating window size 5
Shape: (20231, 50)
Generating window size 10
Shape: (19731, 100)
Generating window size 20
Shape: (18731, 200)
Generating window size 30
Shape: (17731, 300)
Generating window size 40
Shape: (16731, 400)
